<!-- beginner-banner-v2 -->

> 🧭 <strong>비개발자 수강생 안내</strong> — 이 노트북에서 새로 배우는 것: LangChain 핵심 — <code>prompt | model | parser</code> 파이프 조립.
>
> - 📖 강의 페이지: <a href="https://siapapa.github.io/courses/ai-sql-agent/day3/15-langchain-lcel/" target="_blank" rel="noopener noreferrer">day3/15-langchain-lcel</a>
> - 🆕 처음이라면 → <a href="https://siapapa.github.io/courses/ai-sql-agent/beginners-guide/" target="_blank" rel="noopener noreferrer">비개발자 학습 가이드</a>
> - 🔤 모르는 단어 → <a href="https://siapapa.github.io/courses/ai-sql-agent/appendix/glossary/" target="_blank" rel="noopener noreferrer">용어 사전</a>
> - 🛠️ 환경/접속 막힘 → <a href="https://siapapa.github.io/courses/ai-sql-agent/setup/" target="_blank" rel="noopener noreferrer">사전 준비</a> · <a href="https://siapapa.github.io/courses/ai-sql-agent/appendix/troubleshooting/" target="_blank" rel="noopener noreferrer">트러블슈팅</a>
>
> 외부 링크는 새 탭으로 열리도록 설정돼 있어 Colab 의 리디렉션 경고 페이지를 거치지 않습니다.<br/>
> <strong>셀은 위에서 아래로 차례대로 실행</strong>하세요. 시연용 코드(<code>구경만 하세요</code> 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 12. LangChain & LCEL 기초 — 파이프 연산자로 조립하는 체인
> Day 3 · 15H · 소요 약 50분

## 학습 목표

- LangChain 의 핵심 구성요소(`PromptTemplate`, `ChatModel`, `OutputParser`) 를 이해한다.
- **LCEL**(LangChain Expression Language) 의 파이프 연산자(`|`) 로 체인을 조립한다.
- `Runnable` 인터페이스의 `invoke` / `stream` / `batch` 세 가지 실행 모드를 구분한다.
- `RunnablePassthrough` / `RunnableLambda` / `RunnableParallel` 로 체인을 유연하게 변형한다.
- `with_structured_output` 과 `JsonOutputParser` 로 **Pydantic 구조화 출력**을 받는다.

> **DB 없이 돌아갑니다.** 이 노트북은 Neon 연결이 필요 없고 `OPENAI_API_KEY` 만 있으면 됩니다. 13 번(LCEL RAG 체인) 에서 Chroma 를 붙여 완전한 RAG 체인을 만듭니다.


In [ ]:
%pip install -q langchain langchain-openai langchain-core pydantic

In [ ]:
# ============================================================
# 🔁 langchain-core 무결성 자동 점검 + 재시작 (Colab 전용)
# ------------------------------------------------------------
# Colab pip resolver 캐시가 langchain-core 를 부분적으로만 업그레이드해
# 일부 서브모듈이 누락되는 경우, LLM 호출 시 아래 오류가 납니다:
#   ModuleNotFoundError: No module named
#     'langchain_core.messages.block_translators.langchain_v0'
# 실제 실패 코드 경로(_normalize_messages) 를 직접 호출해 보고,
# 같은 ModuleNotFoundError 가 재현되는 경우에만 재설치 + 재시작.
# ============================================================
def _langchain_core_broken() -> bool:
    try:
        from langchain_core.language_models._utils import _normalize_messages
        from langchain_core.messages import HumanMessage
    except Exception:
        return False
    try:
        _normalize_messages([HumanMessage(content="test")])
        return False
    except ModuleNotFoundError as e:
        return "block_translators" in str(e)
    except ImportError as e:
        return "block_translators" in str(e)
    except Exception:
        return False


import sys
if _langchain_core_broken():
    print("⚠️ langchain-core 가 부분 설치 상태입니다 — 강제 재설치합니다.")
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps",
        "langchain-core>=0.3.65,<0.4",
    ])
    if "google.colab" in sys.modules:
        print("🔁 런타임을 자동 재시작합니다. 재시작 후 이 노트북을 처음부터 다시 실행해 주세요.")
        import os
        os.kill(os.getpid(), 9)
    else:
        raise RuntimeError(
            "langchain-core 를 재설치했습니다. 커널을 재시작한 뒤 위에서부터 다시 실행하세요."
        )
else:
    print("✅ langchain-core OK — 다음 셀로 진행해도 됩니다.")

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다.
import os

def _load_secret(key: str, required: bool = True) -> None:
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

_load_secret("OPENAI_API_KEY", required=True)

print("Environment ready.")


## LCEL 은 왜 "파이프" 인가?

```
LCEL = LangChain Expression Language
         │
         ▼
 "Runnable 들을 파이프(|) 로 연결해 체인을 만든다"
         │
         ▼
 모든 컴포넌트가 같은 인터페이스 — invoke · stream · batch
         │
         ▼
 Unix 파이프처럼 자유롭게 조합
```

### 핵심 구성요소

```
┌───────────────┐    ┌───────────────┐    ┌────────────────┐
│ PromptTemplate │ ─► │   ChatModel    │ ─► │ OutputParser   │
│ 변수 삽입     │    │ LLM 호출      │    │ 결과 파싱     │
└───────────────┘    └───────────────┘    └────────────────┘

chain = prompt | model | parser
result = chain.invoke({"var": "value"})
```

이 3 단 체인을 출발점으로, **Runnable 조립**만으로 RAG / 에이전트 / 평가 파이프라인까지 확장할 수 있습니다.


## 1. 기본 체인 — `prompt | model | parser`

가장 단순한 LCEL 체인은 **프롬프트 → LLM → 문자열 파서** 3 단입니다.


In [ ]:
# LCEL 의 가장 단순한 체인 = "프롬프트 → LLM → 파서" 3단.
# 이 한 줄(prompt | model | parser) 이 LangChain 의 핵심 문법입니다.
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# (1) PromptTemplate — 변수 자리표시자({...}) 가 들어간 프롬프트의 "틀".
#     실제 변수 값은 chain.invoke({...}) 시점에 채워집니다.
prompt = ChatPromptTemplate.from_template(
    "당신은 {role} 전문가입니다. 다음 질문에 한국어로 간결히 답변하세요.\n\n질문: {question}"
)
# (2) ChatModel — OpenAI 챗 API 호출 래퍼. temperature=0 으로 매번 같은 답을 유도.
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# (3) OutputParser — LLM 의 응답 객체에서 실제 텍스트만 뽑아 문자열로 반환.
parser = StrOutputParser()

# `|` 연산자로 셋을 연결 → 입력 dict 가 prompt 로 들어가 → LLM 호출 → 응답이 parser 를 통과해 문자열로.
# Unix 파이프(|) 와 같은 발상이라 "앞 단계 출력 → 다음 단계 입력" 흐름이 한눈에 보입니다.
chain = prompt | model | parser

# .invoke() 는 단건 동기 실행. 입력은 prompt 의 변수 이름과 일치하는 dict.
result = chain.invoke({
    "role": "데이터베이스",
    "question": "인덱스는 왜 중요한가요?",
})
print(result)

## 2. 세 가지 실행 모드 — `invoke` · `stream` · `batch`

같은 체인을 **다른 인터페이스**로 실행할 수 있습니다.

- `invoke(inputs)` — 단건 동기 실행
- `stream(inputs)` — 토큰 단위 스트리밍(UX 가 필요한 곳)
- `batch(list_of_inputs)` — 여러 입력을 병렬 실행(평가·튜닝 루프)


In [ ]:
# invoke: 단건
print("[invoke]")
print(chain.invoke({"role": "SQL", "question": "CTE 란 무엇인가요?"})[:200], "...\n")

# stream: 토큰 단위 — UI 스트리밍에 사용
print("[stream] ", end="")
for chunk in chain.stream({"role": "SQL", "question": "윈도우 함수를 간단히 설명해주세요"}):
    print(chunk, end="", flush=True)
print("\n")

# batch: 여러 입력 병렬 실행
print("[batch]")
results = chain.batch([
    {"role": "Python", "question": "리스트와 튜플의 차이?"},
    {"role": "SQL",    "question": "GROUP BY 의 용도?"},
    {"role": "AI",     "question": "RAG 는 무엇인가?"},
])
for i, r in enumerate(results):
    print(f"  batch[{i}]: {r[:80]}...")


## 3. 다양한 `PromptTemplate`

`ChatPromptTemplate.from_messages([...])` 로 **system / human** 역할을 명시적으로 나눌 수 있습니다. `MessagesPlaceholder` 는 대화 히스토리를 넣을 자리입니다(13 번에서 사용).


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {domain} 분야의 전문 분석가입니다. 항상 한국어로 답변하세요."),
    ("human", "{question}"),
])

chain2 = chat_prompt | model | parser
print(chain2.invoke({"domain": "의료", "question": "병원 데이터 분석에서 가장 중요한 포인트는?"}))


## 4. Runnable 유틸 — `Passthrough` · `Lambda` · `Parallel`

- `RunnablePassthrough()` — 입력을 그대로 다음 단계로 흘려 보냄. **딕셔너리의 한 키로 원본 입력을 보존**할 때 자주 쓰입니다.
- `RunnableLambda(fn)` — 임의의 파이썬 함수를 체인에 끼워 넣는 어댑터.
- `RunnableParallel({...})` — 여러 하위 체인을 **동시에** 실행해 키별 결과를 받음.


In [ ]:
# Runnable 어댑터 3 형제 중 둘을 시연 (Parallel 은 다음 셀에서).
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

# RunnablePassthrough() = "받은 값을 그대로 다음 단계로 흘려 보낸다."
#   여기서는 .invoke("문자열") 한 줄로 호출하기 위해 dict 로 변환하는 단계가 첫 노드.
#   {"question": RunnablePassthrough()} 는 입력값을 통째로 question 키에 담는다는 뜻 → RAG 체인의 표준 시작 패턴.
chain_pass = (
    {"question": RunnablePassthrough()}
    | ChatPromptTemplate.from_template("질문: {question}\n한국어로 답변:")
    | model
    | parser
)
print("[Passthrough]", chain_pass.invoke("PostgreSQL 이란 무엇인가요?")[:120], "...\n")

# RunnableLambda(fn) = 임의의 파이썬 함수를 체인에 끼워 넣는 어댑터.
# Python 의 lambda 식은 한 줄짜리 익명 함수 — `lambda x: x.upper()` 는 입력을 대문자로 변환한다는 뜻.
upper = RunnableLambda(lambda x: x.upper())
print("[Lambda]", upper.invoke("hello from lcel"))

In [ ]:
# RunnableParallel — 같은 입력을 두 체인에 동시에 던지고 결과를 딕셔너리로 합침
parallel_chain = RunnableParallel(
    summary=ChatPromptTemplate.from_template("'{topic}' 을 한 문장으로 요약:") | model | parser,
    keywords=ChatPromptTemplate.from_template("'{topic}' 의 핵심 키워드 3 개를 쉼표로 나열:") | model | parser,
)
out = parallel_chain.invoke({"topic": "벡터 데이터베이스"})
print("요약:", out["summary"])
print("키워드:", out["keywords"])


## 5. 구조화 출력 — `with_structured_output`

LLM 이 자유 텍스트를 반환하는 대신 **Pydantic 모델** 로 강제 파싱시키면, 후속 코드가 `.tables` · `.sql` 같은 필드로 바로 접근할 수 있습니다. 20 번(SQL 에이전트) 의 상태 업데이트 · 라우팅 분기에서 이 패턴이 핵심적으로 사용됩니다.


In [ ]:
# 구조화 출력 — LLM 의 자유 텍스트가 아니라 "정의해 둔 dataclass(=Pydantic 모델)" 로 받기.
# 후속 코드가 .tables, .sql 같은 필드로 바로 접근할 수 있어 라우팅·상태 업데이트가 깔끔해집니다.
from pydantic import BaseModel, Field
from typing import Optional


class SQLAnalysis(BaseModel):
    """SQL 쿼리 분석 결과 — Field 의 description 이 LLM 에게 "이 필드에 무엇을 넣어야 하는지" 알려 주는 힌트."""

    tables: list[str] = Field(description="사용해야 할 테이블 목록")
    join_needed: bool = Field(description="JOIN 이 필요한지 여부")
    aggregation: Optional[str] = Field(default=None, description="필요한 집계 함수 (COUNT, SUM 등)")
    difficulty: str = Field(description="난이도: easy, medium, hard")
    sql: str = Field(description="생성된 SQL 쿼리")


# .with_structured_output(클래스) 는 OpenAI tool calling 을 이용해 LLM 응답이 반드시 위 스키마를
# 따르도록 강제합니다. JSON 파싱 실패 / 타입 에러를 자동으로 막아 줍니다.
llm_structured = model.with_structured_output(SQLAnalysis)

prompt_struct = ChatPromptTemplate.from_template(
    """다음 자연어 질문을 분석하여 SQL 쿼리를 생성하세요.

데이터베이스: 병원 (patients, doctors, visits, diagnoses, departments)

질문: {question}"""
)

# 체인 결과의 타입이 SQLAnalysis 인스턴스 — dict 가 아니라 객체이므로 점(.) 으로 필드 접근.
struct_chain = prompt_struct | llm_structured

analysis = struct_chain.invoke({"question": "진료과별 평균 진료비를 보여주세요"})
print("tables     :", analysis.tables)
print("join_needed:", analysis.join_needed)
print("aggregation:", analysis.aggregation)
print("difficulty :", analysis.difficulty)
print("sql        :", analysis.sql)

## 6. `JsonOutputParser` — 다른 LLM 엔진에서도 쓰는 포터블한 대안

`with_structured_output` 은 도구 호출을 지원하는 모델에서 가장 편리하지만, 호환이 제한되는 환경에서는 `JsonOutputParser(pydantic_object=...)` 가 더 안전합니다. 포맷 지시문을 프롬프트에 **자동으로** 삽입해 줍니다.


In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser(pydantic_object=SQLAnalysis)

prompt_json = ChatPromptTemplate.from_template(
    """다음 질문을 분석하세요.
{format_instructions}

질문: {question}"""
).partial(format_instructions=json_parser.get_format_instructions())

json_chain = prompt_json | model | json_parser
parsed = json_chain.invoke({"question": "남성 환자 수는?"})
print("JSON result:", parsed)


## 정리 — LCEL 체크리스트

- `|` 는 "앞 단계의 출력을 뒤 단계의 입력으로" 라는 Unix 파이프와 동일한 발상.
- 딕셔너리 리터럴 `{"a": X, "b": Y}` 도 Runnable 로 간주 → **RunnableParallel 축약형**.
- 복잡한 체인은 결국 **레고 블록** — `Passthrough` · `Lambda` · `Parallel` 3 개가 조립의 대부분을 담당.
- 에이전트/평가 파이프라인에서 **가장 자주 재사용**되는 건 `with_structured_output` — 파이프로 흘러온 결과를 바로 파싱해 다음 단계의 **상태**로 만들 수 있기 때문입니다.


## 실습 과제

다음 1 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. SQL 생성 체인으로 병원 DB 질문 3개 테스트
`with_structured_output`을 활용하여 병원 DB 질문 3개를 테스트합니다.

아래 3개 질문을 위에서 만든 `struct_chain` 으로 실행하고, 반환된 `SQLAnalysis` 객체의 `tables / join_needed / aggregation / difficulty / sql` 을 출력하세요.

1. 전체 환자 수는 몇 명인가요?
2. 진료과별 의사 수를 보여주세요
3. 지난달 응급 진료 건수와 평균 비용은?

_힌트: `struct_chain.invoke({"question": q})` 의 반환값은 Pydantic 객체이므로 점 표기법(`result.tables`, `result.sql` 등)으로 필드에 접근할 수 있습니다._


In [ ]:
# ============================================================
# 실습 과제 — SQL 생성 체인으로 병원 DB 질문 3개 테스트
# ============================================================

# 실습 1: SQL 생성 체인으로 병원 DB 질문 3개 테스트
# TODO: struct_chain.invoke({"question": q}) 로 3개 질문을 돌리고 result.tables / join_needed / aggregation / difficulty / sql 을 출력하세요.
# 여기에 구현하세요.


## 다음 노트북에서는…

**`13_lcel_rag_chain.ipynb`** 에서 방금 익힌 LCEL 에 **ChromaDB Retriever** 를 붙여 완전한 RAG 체인을 조립합니다.  `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | parser` 패턴, 스트리밍 출력, `with_fallbacks` 를 이용한 이중 LLM 안전망, 그리고 **대화 히스토리** 를 끼워 넣는 멀티턴 RAG 까지 한 번에 정리합니다.
